In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime

import polars as pl
from IPython.display import display

from fintl.accounts_etl.common.schemas import Config

In [ ]:
config = Config()

In [ ]:
path_root = config.target_dir

In [ ]:
path_balances = path_root / "all-balances.parquet"
path_balances.exists()

In [ ]:
balances = pl.read_parquet(path_balances)
balances.head()

In [ ]:
len(balances)

In [ ]:
balances = balances.with_columns(
    **{
        "name": pl.col("provider").str.to_lowercase()
        + " "
        + pl.col("service").str.to_lowercase()
    }
)
balances.head()

Accounts over time

In [ ]:
chart = balances.plot.scatter(x="date", y="amount", color="name").properties(
    width=600, height=400
)
chart

In [ ]:
tmp = (
    balances.sort("date")
    .group_by("name", pl.col("date").dt.month_end(), maintain_order=True)
    .agg(pl.col("amount").last())
)
tmp.filter(pl.col("name").eq("dkb giro"))

In [ ]:
name = "dkb giro"
t = datetime.date(2025, 1, 1)
changes = (
    tmp.filter(pl.col("name").eq(name))
    .with_columns(pl.col("amount").diff().alias("change"))
    .filter(pl.col("date").ge(t))
)

with pl.Config(set_tbl_rows=42):
    display(changes.tail(21))

changes["change"].median()

In [ ]:
recent_balances = balances.filter(
    pl.col("date").ge(datetime.date(2024, 8, 25)) & pl.col("name").eq("DKB giro")
)
recent_balances.head()

In [ ]:
recent_balances.plot.scatter(x="date", y="amount", color="name")

In [ ]:
path_transactions = path_root / "all-transactions.parquet"
path_transactions.exists()

In [ ]:
transactions = pl.read_parquet(path_transactions)
transactions.head()

In [ ]:
len(transactions)